# Study 1: Colab T4 tiny-LoRA training gate

This notebook is a thin launcher for the repository implementation. It proves one real optimiser step, a complete checkpoint resume to step two, and an independent PEFT adapter reload on the locked 20-source training subset.

Use Colab runtime version **2025.07**, select a **T4 GPU**, expose the read-capable `HF_TOKEN` secret, paste the full pushed Git commit below, and run all cells. The resulting bundle is infrastructure evidence, not a paper model.

In [ ]:
from pathlib import Path
import re

REPOSITORY_URL = "https://github.com/dizza01/VLM.git"
REPOSITORY_COMMIT = "PASTE_FULL_40_CHARACTER_COMMIT_SHA_HERE"
CHECKOUT_ROOT = Path("/content/vlm-training-gate")
INSTALL_SOURCE = Path("/content/gi-vqa-training-gate-install")
WORK_DIR = Path("/content/gi_vqa_training_gate_work")
ARTIFACT_DIR = Path("/content/gi_vqa_training_gate_artifacts")
DOWNLOAD_BUNDLE = True

if re.fullmatch(r"[0-9a-fA-F]{40}", REPOSITORY_COMMIT) is None:
    raise ValueError("REPOSITORY_COMMIT must be the full 40-character SHA of the pushed commit")
REPOSITORY_COMMIT = REPOSITORY_COMMIT.lower()
PROJECT_ROOT = CHECKOUT_ROOT / "gi_vqa_research"
CONFIG_PATH = PROJECT_ROOT / "configs/study1/smoke.yaml"
SPLIT_MANIFEST = PROJECT_ROOT / "protocols/study1/grouped_split_manifest.json"
IMAGE_CACHE_MANIFEST = PROJECT_ROOT / "protocols/study1/smoke_training_image_cache_manifest.json"
print(f"Training-gate commit: {REPOSITORY_COMMIT}")

In [ ]:
import hashlib
import json
import platform
import shutil
import subprocess
import sys
import traceback

def run_command(command, *, cwd=None, capture=False):
    return subprocess.run(
        [str(part) for part in command], cwd=cwd, check=True,
        text=True, capture_output=capture,
    )

for path in (CHECKOUT_ROOT, INSTALL_SOURCE, WORK_DIR, ARTIFACT_DIR):
    if path.exists():
        shutil.rmtree(path)
ARTIFACT_DIR.mkdir(parents=True)
BOOTSTRAP_OK = False

try:
    if sys.version_info[:2] != (3, 11):
        raise RuntimeError(
            "Python 3.11 is required. Select Colab runtime version 2025.07, reconnect, "
            f"and run from the top. Observed: {sys.version}"
        )
    gpu_query = run_command(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader,nounits"],
        capture=True,
    ).stdout.strip().splitlines()
    if len(gpu_query) != 1 or "T4" not in gpu_query[0]:
        raise RuntimeError(f"Exactly one T4 GPU is required; observed {gpu_query}")

    run_command(["git", "clone", "--filter=blob:none", "--no-checkout", REPOSITORY_URL, CHECKOUT_ROOT])
    run_command(["git", "checkout", "--detach", REPOSITORY_COMMIT], cwd=CHECKOUT_ROOT)
    resolved_commit = run_command(["git", "rev-parse", "HEAD"], cwd=CHECKOUT_ROOT, capture=True).stdout.strip().lower()
    if resolved_commit != REPOSITORY_COMMIT:
        raise RuntimeError(f"Resolved commit {resolved_commit} does not match {REPOSITORY_COMMIT}")
    for required_path in (PROJECT_ROOT, CONFIG_PATH, SPLIT_MANIFEST, IMAGE_CACHE_MANIFEST):
        if not required_path.exists():
            raise RuntimeError(f"Checked-out commit is missing: {required_path}")

    shutil.copytree(PROJECT_ROOT, INSTALL_SOURCE)
    run_command([sys.executable, "-m", "pip", "install", "--no-cache-dir", f"{INSTALL_SOURCE}[gpu]"])
    run_command([
        sys.executable, "-m", "pip", "install", "--no-cache-dir",
        "--force-reinstall", "--no-deps", str(INSTALL_SOURCE),
    ])
    pip_check = subprocess.run(
        [sys.executable, "-m", "pip", "check"], check=False,
        capture_output=True, text=True,
    )
    pip_check_text = "\n".join(
        part.strip() for part in (pip_check.stdout, pip_check.stderr) if part.strip()
    )
    (ARTIFACT_DIR / "pip_check.txt").write_text(
        (pip_check_text + "\n") if pip_check_text else "No broken requirements found.\n",
        encoding="utf-8",
    )
    scoped = {
        "accelerate", "bitsandbytes", "datasets", "gi-vqa-research",
        "huggingface-hub", "ms-swift", "numpy", "peft", "pillow",
        "pyyaml", "sentencepiece", "torch", "transformers", "wandb",
    }
    scoped_conflicts = [
        line for line in pip_check_text.splitlines()
        if line and line.split(maxsplit=1)[0].lower().replace("_", "-") in scoped
    ]
    if scoped_conflicts:
        raise RuntimeError("Project-stack dependency conflicts: " + " | ".join(scoped_conflicts))
    if pip_check.returncode != 0:
        print("Global pip check WARNING (unrelated Colab packages):")
        print(pip_check_text)

    installed_path = Path(run_command(
        [sys.executable, "-c", "import gi_vqa.training_gate; print(gi_vqa.training_gate.__file__)"],
        capture=True,
    ).stdout.strip())
    source_path = PROJECT_ROOT / "src/gi_vqa/training_gate.py"
    installed_sha = hashlib.sha256(installed_path.read_bytes()).hexdigest()
    source_sha = hashlib.sha256(source_path.read_bytes()).hexdigest()
    if installed_sha != source_sha:
        raise RuntimeError("Installed training gate differs from the verified checkout")
    git_status = run_command(
        ["git", "status", "--porcelain=v1", "--untracked-files=all"],
        cwd=CHECKOUT_ROOT, capture=True,
    ).stdout.strip()
    if git_status:
        raise RuntimeError(f"Verified checkout became dirty: {git_status.splitlines()}")

    bootstrap = {
        "status": "PASS", "python": sys.version,
        "implementation": platform.python_implementation(),
        "gpu_query": gpu_query, "repository_url": REPOSITORY_URL,
        "repository_commit": resolved_commit, "checkout_clean": True,
        "installed_training_gate_path": str(installed_path),
        "installed_training_gate_sha256": installed_sha,
        "global_pip_check_exit_code": pip_check.returncode,
        "scoped_pip_conflicts": scoped_conflicts,
    }
    (ARTIFACT_DIR / "bootstrap_environment.json").write_text(
        json.dumps(bootstrap, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    BOOTSTRAP_OK = True
    print("Bootstrap PASS: exact clean checkout and pinned GPU stack are ready.")
except BaseException as exc:
    failure = {
        "status": "FAIL", "phase": "bootstrap",
        "type": f"{type(exc).__module__}.{type(exc).__name__}",
        "message": re.sub(r"\b(?:hf|api)_[A-Za-z0-9_-]{12,}\b", "<redacted-token>", str(exc)),
        "traceback": re.sub(r"\b(?:hf|api)_[A-Za-z0-9_-]{12,}\b", "<redacted-token>", traceback.format_exc()),
    }
    (ARTIFACT_DIR / "bootstrap_failure.json").write_text(
        json.dumps(failure, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    print(f"Bootstrap FAIL: {failure['message']}")

In [ ]:
import os

os.environ["HF_HOME"] = "/content/huggingface-cache"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_MODE"] = "disabled"
AUTH_OK = False
HF_USERNAME = None

if not BOOTSTRAP_OK:
    print("Authentication SKIPPED because bootstrap failed.")
else:
    try:
        from google.colab import userdata
        from huggingface_hub import login, whoami

        hf_token = userdata.get("HF_TOKEN")
        if not isinstance(hf_token, str) or not hf_token.strip():
            raise RuntimeError("Colab secret HF_TOKEN is missing or notebook access is disabled")
        login(token=hf_token, add_to_git_credential=False)
        identity = whoami(token=hf_token)
        HF_USERNAME = identity.get("name") or identity.get("fullname")
        os.environ["HF_TOKEN"] = hf_token
        del hf_token
        AUTH_OK = True
        print(f"Hugging Face authentication PASS for user: {HF_USERNAME}")
    except BaseException as exc:
        message = re.sub(r"\b(?:hf|api)_[A-Za-z0-9_-]{12,}\b", "<redacted-token>", str(exc))
        failure = {
            "status": "FAIL", "phase": "authentication",
            "type": f"{type(exc).__module__}.{type(exc).__name__}",
            "message": message,
        }
        (ARTIFACT_DIR / "authentication_failure.json").write_text(
            json.dumps(failure, indent=2, sort_keys=True) + "\n", encoding="utf-8"
        )
        print(f"Authentication FAIL: {message}")

In [ ]:
gate_command = [
    sys.executable, "-m", "gi_vqa.training_gate",
    "--config", str(CONFIG_PATH),
    "--repository-root", str(CHECKOUT_ROOT),
    "--expected-commit", REPOSITORY_COMMIT,
    "--split-manifest", str(SPLIT_MANIFEST),
    "--image-cache-manifest", str(IMAGE_CACHE_MANIFEST),
    "--work-dir", str(WORK_DIR),
    "--artifact-dir", str(ARTIFACT_DIR),
    "--required-gpu-substring", "T4",
]
REPORT_PATH = ARTIFACT_DIR / "training_gate_report.json"
if BOOTSTRAP_OK and AUTH_OK:
    GATE_PROCESS = subprocess.run(gate_command, check=False)
else:
    GATE_PROCESS = subprocess.CompletedProcess(gate_command, returncode=1)
    failed_phase = "bootstrap" if not BOOTSTRAP_OK else "authentication"
    source_failure = ARTIFACT_DIR / f"{failed_phase}_failure.json"
    early = json.loads(source_failure.read_text(encoding="utf-8")) if source_failure.is_file() else {}
    REPORT_PATH.write_text(
        json.dumps({
            "schema_version": "gi-vqa-colab-t4-training-gate-v1",
            "status": "FAIL", "phase": failed_phase, "checks": {},
            "artifact_paths": {"report": str(REPORT_PATH)},
            "error": {
                "type": "EarlyTrainingGateFailure",
                "message": early.get("message", f"Gate skipped because {failed_phase} failed"),
            },
        }, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
if REPORT_PATH.is_file():
    GATE_REPORT = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
else:
    GATE_REPORT = {
        "status": "FAIL", "checks": {},
        "error": {"type": "MissingReport", "message": "Gate process produced no report"},
    }
    REPORT_PATH.write_text(json.dumps(GATE_REPORT, indent=2) + "\n", encoding="utf-8")
print(f"Training-gate process exit code: {GATE_PROCESS.returncode}")
print(f"Training-gate status: {GATE_REPORT['status']}")
print(f"Machine-readable report: {REPORT_PATH}")

In [ ]:
from IPython.display import Markdown, display

checks = GATE_REPORT.get("checks", {})
passed = sum(bool(value.get("passed")) for value in checks.values())
display(Markdown(
    f"## Training gate {GATE_REPORT['status']}  \n"
    f"Recorded checks passed: **{passed}/{len(checks)}**"
))
if GATE_REPORT.get("status") == "PASS":
    resume = GATE_REPORT["training"]["resume_verification"]
    reload_probe = GATE_REPORT["training"]["adapter_reload"]
    print(f"Training items/source images: {GATE_REPORT['training_subset']['records']}")
    print(f"Checkpoint resume: step {resume['resumed_from_global_step']} → {resume['finished_global_step']}")
    print(f"Adapter changed after resume: {resume['adapter_changed_after_resume']}")
    print(f"Independent adapter-reload loss: {reload_probe['loss']:.8g}")
    print(f"Peak reload-probe CUDA memory: {reload_probe['peak_cuda_memory_bytes'] / 2**30:.2f} GiB")
else:
    error = GATE_REPORT.get("error", {})
    print(f"Failure type: {error.get('type', 'unknown')}")
    print(f"Failure message: {error.get('message', 'No message recorded')}")
    failed = [name for name, value in checks.items() if not value.get('passed')]
    print(f"Failed checks: {failed}")

In [ ]:
import zipfile

os.environ.pop("HF_TOKEN", None)
pip_freeze = subprocess.run(
    [sys.executable, "-m", "pip", "freeze", "--all"],
    check=True, capture_output=True, text=True,
).stdout
(ARTIFACT_DIR / "pip_freeze.txt").write_text(pip_freeze, encoding="utf-8")
bundle_members = [
    path for path in sorted(ARTIFACT_DIR.iterdir())
    if path.is_file() and path.name != "training_gate_bundle.zip"
]
member_hashes = {
    path.name: hashlib.sha256(path.read_bytes()).hexdigest()
    for path in bundle_members
}
bundle_manifest = {
    "training_gate_status": GATE_REPORT["status"],
    "repository_commit": REPOSITORY_COMMIT,
    "members_sha256": member_hashes,
}
manifest_path = ARTIFACT_DIR / "bundle_manifest.json"
manifest_path.write_text(
    json.dumps(bundle_manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
bundle_members.append(manifest_path)
BUNDLE_PATH = ARTIFACT_DIR / "training_gate_bundle.zip"
with zipfile.ZipFile(BUNDLE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in bundle_members:
        archive.write(path, arcname=path.name)
BUNDLE_SHA256 = hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest()
print(f"Evidence bundle: {BUNDLE_PATH}")
print(f"Bundle SHA-256: {BUNDLE_SHA256}")
if DOWNLOAD_BUNDLE:
    from google.colab import files
    files.download(str(BUNDLE_PATH))

In [ ]:
if GATE_PROCESS.returncode != 0 or GATE_REPORT.get("status") != "PASS":
    raise RuntimeError(
        "Training gate FAILED. Preserve the evidence bundle and fix its recorded failure "
        "before larger training or the 20-item adapter smoke run."
    )
print(
    "Training gate PASS. Preserve the evidence bundle with this Git commit. "
    "Next gate: run the restart-safe 20-item development inference and explanation pipeline."
)